# Climate Intelligence Engine
Engine analítica para detecção de inconsistências, completamento de dados e otimização de precisão climática.

## 1. Imports e Configuração

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Scikit-learn — modelos clássicos
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# PyTorch — redes densas
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Dados climáticos
import xarray as xr

# Configurações visuais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Ambiente pronto.')

## 2. Carga de Dados

In [ ]:
# --- Substitua este bloco pelo carregamento real dos seus dados ---
# Exemplo com NetCDF:  ds = xr.open_dataset(DATA_DIR / 'clima.nc')
# Exemplo com CSV:     df = pd.read_csv(DATA_DIR / 'clima.csv', parse_dates=['date'])

# Dados sintéticos para desenvolvimento e testes
n_samples = 2000
dates = pd.date_range('2015-01-01', periods=n_samples, freq='D')

df = pd.DataFrame({
    'date': dates,
    'temp_max':   20 + 10 * np.sin(2 * np.pi * np.arange(n_samples) / 365) + np.random.normal(0, 2, n_samples),
    'temp_min':   10 + 8  * np.sin(2 * np.pi * np.arange(n_samples) / 365) + np.random.normal(0, 1.5, n_samples),
    'humidity':   60 + 20 * np.cos(2 * np.pi * np.arange(n_samples) / 365) + np.random.normal(0, 5, n_samples),
    'wind_speed': np.abs(np.random.normal(15, 5, n_samples)),
    'pressure':   1013 + np.random.normal(0, 8, n_samples),
    'rainfall':   np.abs(np.random.exponential(3, n_samples)),
}).set_index('date')

print(df.shape)
df.head()

## 3. Análise Exploratória

In [ ]:
df.describe()

In [ ]:
# Valores ausentes
missing = df.isnull().sum()
print('Valores ausentes:\n', missing[missing > 0] if missing.any() else 'Nenhum')

# Correlação
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

In [ ]:
# Série temporal
df[['temp_max', 'temp_min']].resample('ME').mean().plot(title='Temperatura Média Mensal')
plt.ylabel('°C')
plt.tight_layout()
plt.show()

## 4. Preparação dos Dados

In [ ]:
# Target: prever temp_max do dia seguinte
FEATURES = ['temp_min', 'humidity', 'wind_speed', 'pressure', 'rainfall']
TARGET = 'temp_max'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Treino: {X_train_s.shape} | Teste: {X_test_s.shape}')

## 5. Treinamento dos Modelos

### 5.1 Regressão Linear

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)

print('Linear Regression')
print(f'  MAE:  {mean_absolute_error(y_test, pred_lr):.3f}')
print(f'  RMSE: {mean_squared_error(y_test, pred_lr, squared=False):.3f}')
print(f'  R²:   {r2_score(y_test, pred_lr):.3f}')

### 5.2 K-Nearest Neighbors

In [ ]:
knn = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn.fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)

print('KNN')
print(f'  MAE:  {mean_absolute_error(y_test, pred_knn):.3f}')
print(f'  RMSE: {mean_squared_error(y_test, pred_knn, squared=False):.3f}')
print(f'  R²:   {r2_score(y_test, pred_knn):.3f}')

### 5.3 Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf.fit(X_train_s, y_train)
pred_rf = rf.predict(X_test_s)

print('Random Forest')
print(f'  MAE:  {mean_absolute_error(y_test, pred_rf):.3f}')
print(f'  RMSE: {mean_squared_error(y_test, pred_rf, squared=False):.3f}')
print(f'  R²:   {r2_score(y_test, pred_rf):.3f}')

# Importância das features
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', title='Feature Importance — Random Forest')
plt.tight_layout()
plt.show()

### 5.4 Rede Densa (PyTorch)

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),           nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_loader(X, y, batch_size=64, shuffle=True):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = DenseNet(len(FEATURES)).to(DEVICE)
optim  = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

train_loader = make_loader(X_train_s, y_train)
test_loader  = make_loader(X_test_s, y_test, shuffle=False)

EPOCHS = 50
train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optim.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optim.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Loss: {train_losses[-1]:.4f}')

# Avaliação
model.eval()
with torch.no_grad():
    pred_nn = model(torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()

print('\nDense Network')
print(f'  MAE:  {mean_absolute_error(y_test, pred_nn):.3f}')
print(f'  RMSE: {mean_squared_error(y_test, pred_nn, squared=False):.3f}')
print(f'  R²:   {r2_score(y_test, pred_nn):.3f}')

## 6. Comparação dos Modelos

In [ ]:
results = pd.DataFrame([
    {'Model': 'Linear Regression', 'MAE': mean_absolute_error(y_test, pred_lr),  'R2': r2_score(y_test, pred_lr)},
    {'Model': 'KNN',               'MAE': mean_absolute_error(y_test, pred_knn), 'R2': r2_score(y_test, pred_knn)},
    {'Model': 'Random Forest',     'MAE': mean_absolute_error(y_test, pred_rf),  'R2': r2_score(y_test, pred_rf)},
    {'Model': 'Dense Network',     'MAE': mean_absolute_error(y_test, pred_nn),  'R2': r2_score(y_test, pred_nn)},
]).set_index('Model').round(3)

print(results.sort_values('MAE'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results['MAE'].sort_values().plot(kind='barh', ax=axes[0], title='MAE (menor é melhor)', color='steelblue')
results['R2'].sort_values().plot(kind='barh',  ax=axes[1], title='R² (maior é melhor)',  color='seagreen')
plt.tight_layout()
plt.show()